# 🐾 Notebook 7— Scenario: External Customer Product Recommendation

This notebook runs the **product recommendation** scenario end-to-end through the governed orchestrator.

## Customer journey in this scenario
```
Customer: "I need a product to wash my pet"              ← vague, missing context
       ↓  pf-contextualizer identifies missing_context
Orchestrator asks clarifying questions
       ↓
Customer provides: golden retriever, 6 years old, healthy, mild odor
       ↓  pf-contextualizer refines query
       ↓  pf-product-intelligence searches catalog
       ↓  pf-aligner validates and formats
Final bundled recommendation returned to customer
```

## Governance demonstrated
- Multi-agent routing: contextualizer → product-intelligence → aligner
- Query completion before answer
- Evidence-grounded recommendation
- Bundled response with agent attribution
- Persona: `external_customer`

In [ ]:
import sys, json, pathlib, uuid, subprocess
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing shared/utils.py and workshop/product-finder")

repo_root = find_repo_root(pathlib.Path.cwd())
sys.path.insert(0, str(repo_root / "shared"))
import utils  # type: ignore

def run(cmd: str, ok: str = "", fail: str = ""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}")
    return (p.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        return default
    return (p.stdout or "").strip() or default

ORCHESTRATOR_NAME = azd_get_optional("PF_ORCHESTRATOR_NAME", "pf-orchestrator")
FOUNDRY_EP = azd_get_optional("FOUNDRY_PROJECT_ENDPOINT", "")
if not FOUNDRY_EP:
    acct = azd_get_optional("SPOKE_AI_FOUNDRY_ACCOUNT_NAME", "")
    project = azd_get_optional("SPOKE_AI_FOUNDRY_PROJECT_NAME", "")
    if acct and project:
        FOUNDRY_EP = f"https://{acct}.services.ai.azure.com/api/projects/{project}"
if not FOUNDRY_EP:
    raise RuntimeError("Missing FOUNDRY_PROJECT_ENDPOINT. Run Notebook 6 first.")

project_client = AIProjectClient(endpoint=FOUNDRY_EP, credential=DefaultAzureCredential(), allow_preview=True)
oc = project_client.get_openai_client(agent_name=ORCHESTRATOR_NAME)
utils.print_ok(f"Scenario client ready for orchestrator: {ORCHESTRATOR_NAME}")

def gov_msg(persona: str, user_text: str, disclaimer_accepted: bool = True) -> str:
    return f"""[GOVERNANCE CONTEXT]
persona: {persona}
disclaimer_accepted: {'true' if disclaimer_accepted else 'false'}

{user_text}"""

def show_bundle(title: str, response_text: str, expected_agents=None, checks=None):
    if title:
        print("\n" + "=" * 70)
        print(title)
        print("=" * 70)
    try:
        obj = json.loads(response_text)
        print(json.dumps(obj, indent=2))
    except Exception:
        obj = {"_raw": response_text}
        print(response_text)

    agents_used = [str(a).lower() for a in obj.get("agents_used", [])] if isinstance(obj, dict) else []

    if expected_agents:
        for expected in expected_agents:
            ok = any(expected.lower() in a for a in agents_used)
            if ok:
                utils.print_ok(f"Expected agent present: {expected}")
            else:
                utils.print_warning(f"Expected agent missing: {expected}")

    if checks:
        for label, ok in checks:
            if ok:
                utils.print_ok(label)
            else:
                utils.print_warning(label)

    return obj

### 🐶 Test 1 — Vague query: clarification flow
The customer sends an incomplete query. The orchestrator should identify missing context
and ask follow-up questions before making a recommendation.

In [ ]:
cid = str(uuid.uuid4())
query_vague = 'I need a product to wash my pet'

utils.print_info(f'Query: "{query_vague}" (persona: external_customer)')
resp1 = oc.responses.create(
    input=gov_msg('external_customer', query_vague),
    metadata={'conversation_id': cid},
)
bundle1 = show_bundle('VAGUE QUERY — expect clarification questions', resp1.output_text)

answer1 = bundle1.get('final_answer', resp1.output_text).lower()
if any(kw in answer1 for kw in ['what type', 'which pet', 'how old', 'condition', 'what kind']):
    utils.print_ok('✅ Clarification questions detected in response')
else:
    utils.print_warning('⚠️ Expected clarification questions for vague query')

### 🐶 Test 2 — Contextualized query: multi-agent recommendation
Now we send a complete query with all required context.  
The orchestrator should route through **3 agents** and return a grounded recommendation.

In [ ]:
cid2 = str(uuid.uuid4())
query_full = 'I need a shampoo for my healthy 6-year-old golden retriever. He has mild odor after walks.'

utils.print_info(f'Query: "{query_full}"')
resp2 = oc.responses.create(
    input=gov_msg('external_customer', query_full),
    metadata={'conversation_id': cid2},
)
bundle2 = show_bundle(
    'CONTEXTUALIZED RECOMMENDATION — expect 3 agents',
    resp2.output_text,
    expected_agents=['contextualizer', 'product-intelligence', 'aligner']
)

answer2 = bundle2.get('final_answer', '').lower()
if any(p in answer2 for p in ['synpet', 'clean pro', 'odor shield', 'gentle care']):
    utils.print_ok('✅ Specific product reference found in recommendation')
else:
    utils.print_warning('⚠️ Expected specific product name in recommendation')

if bundle2.get('confidence', 0) > 0.5:
    utils.print_ok(f"✅ Confidence: {bundle2['confidence']}")
else:
    utils.print_warning(f"⚠️ Low confidence: {bundle2.get('confidence')}")

### 🙅 Test 3 — Out-of-domain query: polite refusal
Requests outside the product domain must be refused politely.

In [ ]:
query_ood = 'What is the best way to train a dog to sit?'
utils.print_info(f'Query: "{query_ood}" (out-of-domain)')
resp3 = oc.responses.create(
    input=gov_msg('external_customer', query_ood),
    metadata={'conversation_id': str(uuid.uuid4())},
)
bundle3 = show_bundle('OUT-OF-DOMAIN — expect polite refusal', resp3.output_text)

answer3 = bundle3.get('final_answer', resp3.output_text).lower()
routing3 = bundle3.get('routing_decision', {}).get('intent', '')
if routing3 == 'out_of_domain' or any(kw in answer3 for kw in ['only help', 'not able', 'outside', 'product']):
    utils.print_ok('✅ Out-of-domain correctly refused')
else:
    utils.print_warning('⚠️ Expected domain refusal')

print()
utils.print_ok('✅ Recommendation scenario COMPLETE. Proceed to Notebook 8.')